In [74]:
import pandas as pd
import numpy as np

In [178]:
# df = pd.read_csv('data/B208_2022-10-09_03-26-03_2022-10-09_22-56-13.csv')
df = pd.read_csv('data/B208_2021-09-25_05-59-40_2021-09-25_18-15-31.csv')

In [179]:
# convert radians to degrees
df['latitude'] = df['gnss_latitude'] * 180.0 / np.pi
df['longitude'] = df['gnss_longitude'] * 180.0 / np.pi
# convert temperature to celsius
df['temp_celsius'] = df['temperature_ambient'] - 273.15
# Convert to datetime and strip the +00:00 timezone
df['timestamp'] = pd.to_datetime(df['time_iso']).dt.tz_localize(None)
# add an extra column that shows the speed in km
df['speed_kmh'] = df['odometry_vehicleSpeed'] * 3.6

# convert compass course to degrees and If speed is near zero on Bus 208, keep the last known heading instead of snapping to 0 (useful for UI later)
df['heading_deg'] = df['gnss_course'] * 180.0 / np.pi
df.loc[df['odometry_vehicleSpeed'] < 0.5, 'heading_deg'] = np.nan
df['heading_deg'] = df['heading_deg'].ffill().fillna(0).round(1)

# create a column with power in Kilowatts
df['power_kw'] = (df['electric_powerDemand'] / 1000.0).round(2)

# brake pressure in bars
df['brake_bar'] = (df['traction_brakePressure'] / 100000.0).round(2)

# ffill passenger count so they persist while driving
df['itcs_numberOfPassengers'] = df['itcs_numberOfPassengers'].ffill().fillna(0).astype(int)


df = df.drop(columns=['gnss_latitude', 'gnss_longitude', 'temperature_ambient', 'time_iso', 'gnss_course'])
df.sample(100)
len(df)
df.sample(10)

,time_unix,electric_powerDemand,gnss_altitude,itcs_busRoute,itcs_numberOfPassengers,itcs_stopName,odometry_articulationAngle,odometry_steeringAngle,odometry_vehicleSpeed,odometry_wheelSpeed_fl,...,traction_brakePressure,traction_tractionForce,latitude,longitude,temp_celsius,timestamp,speed_kmh,heading_deg,power_kw,brake_bar
25938,1632575518,14432.400,417.3000,-,10,-,0.043633,-0.008727,0.586098,0.072630,...,62358.16,-8478.368000,47.377674,8.498408,26.0,2021-09-25 13:11:58,2.109955,120.1,14.43,0.62
30389,1632579969,10615.660,474.5063,-,8,-,0.034907,0.000000,10.520610,10.526520,...,0.00,-11.014620,47.398100,8.536332,27.0,2021-09-25 14:26:09,37.874196,76.0,10.62,0.00
34229,1632583809,10856.120,480.5082,-,9,-,0.034907,0.000000,0.000000,0.000000,...,245000.00,0.000000,47.397464,8.541512,26.0,2021-09-25 15:30:09,0.000000,23.7,10.86,2.45
21555,1632571135,7066.770,415.8855,-,21,-,0.043633,0.024291,0.000000,0.000000,...,243333.30,0.000000,47.381541,8.492037,25.0,2021-09-25 11:58:55,0.000000,32.0,7.07,2.43
12199,1632561779,3961.151,415.9000,-,16,-,0.043633,0.008727,-0.001916,0.000000,...,255244.60,0.000000,47.390875,8.522793,20.0,2021-09-25 09:22:59,-0.006899,33.5,3.96,2.55
11912,1632561492,4610.087,411.0000,-,26,-,0.026180,0.008727,9.799040,9.787197,...,0.00,8.297047,47.380779,8.512904,19.0,2021-09-25 09:18:12,35.276544,32.0,4.61,0.00
40127,1632589707,22118.620,401.8066,-,6,-,0.043633,0.000000,7.402894,7.407516,...,0.00,1992.302000,47.388657,8.489877,24.0,2021-09-25 17:08:27,26.650418,118.1,22.12,0.00
27498,1632577078,6606.733,484.1000,-,4,-,0.034907,0.003142,0.000000,0.000000,...,245300.00,0.000000,47.397710,8.541643,26.0,2021-09-25 13:37:58,0.000000,22.3,6.61,2.45
5677,1632555257,14632.190,485.0859,-,8,-,0.043633,0.026180,0.000000,0.000000,...,250833.30,0.000000,47.397510,8.541523,15.0,2021-09-25 07:34:17,0.000000,23.7,14.63,2.51
10157,1632559737,-15487.310,414.7000,-,26,-,0.052360,0.000000,2.051412,1.980320,...,130512.20,-11465.250000,47.390691,8.522335,19.0,2021-09-25 08:48:57,7.385083,213.2,-15.49,1.31


In [180]:
# Slice out just the 7:00 AM to 9:00 AM window for replay engine
# df_demo = df[(df['timestamp'].dt.hour >= 7) & (df['timestamp'].dt.hour <= 8)]

df.to_csv('data/train_mission.csv', index=False)


In [171]:
# df_demo.to_csv('data/clean_replay_mission.csv', index=False)
# df_demo.head(100).to_json('data/mock_state.json', orient='records', indent=2)

# len(df_demo)

7200